# Reproducing Lesica et al. (2007) from dandiset 002028

**Paper.** Lesica NA, Jin J, Weng C, Yeh C-I, Butts DA, Stanley GB, Alonso J-M (2007). Adaptation to
stimulus contrast and correlations during natural visual stimulation. *Neuron* 55:479-491.
[doi:10.1016/j.neuron.2007.07.013](https://doi.org/10.1016/j.neuron.2007.07.013). The claims file
(`claims/lesica-2007.yaml`) was written from the version of record and its Supplemental Data.

**Dandiset.** [DANDI:002028](https://dandiarchive.org/dandiset/002028/0.261102.1631), version
0.261102.1631. The paper cites no version (it predates the dandiset, created in 2026). This is the
only published version and is pinned; the draft has the same 33 assets and size.

**What the dandiset holds.** Cat LGN single units from four animals (Expt1 to Expt4), with a
stimulus library of the movie and noise frames. The paper's data are in Expt1 and Expt2 (see the
data check): the 15 min movie at two contrasts, the long binary noise and the 12 s repeats (Expt1),
and the 90 s repeats (Expt2). Expt3 and Expt4 belong to the dandiset's other papers.

**Method in brief.** RFs (21 x 21 pixels, 12 lags of one frame) are estimated as the mean of nine
segment estimates, by the batch ridge solution that the paper's recursive least squares converges
to, with the same number of spikes at high and low contrast; the penalty is chosen by the paper's
rule (held-out prediction of the movie), and the paper's literal penalty and a lower one are run as
variants. Temporal width, latency and biphasic ratio come from RFs at the paper's resolution, 128 Hz
bins and 24 lags of 8 ms, in a 15 x 15 window. NLs from the filtered stimulus in 8 ms bins, groups
of 250; a half-wave rectifier fit gives gain and offset. Reliability and sparseness
come from the repeats. Every parameter is cited in `lesica.py`; the choices the paper leaves open
are listed in the attempt record as deviations.

| Claim | Tested here |
|---|---|
| sample-composition | tested |
| firing-rate-contrast, firing-rate-lc-ns-wn-similar | tested |
| ln-prediction-accuracy | tested |
| fig3b, fig3c, fig3d (six claims) | tested |
| surround-temporal-correlation, figs1-property-correlations | tested |
| fig4b, fig4c, fig4d (six claims) | tested |
| fig5-within-trial-contrast | tested (OFF-center Y cells) |
| fig6a, fig6b (four claims) | tested |
| fig7c-effective-contrast, fig7d-reliability-tracks-effective-contrast | tested |
| fig8a-filtered-kurtosis, fig8-model-sparseness-shuffle, fig8b, fig8de | tested |
| disc-lcns-hcwn-reliability-sparseness, disc-lcns-hcwn-rf-nl-differences | tested |
| methods-latency-biphasic, methods-offset-varies-across-animals | tested |
| celltype claims (four) | tested |
| methods-spherical-asymmetry | tested |

In [ ]:
import json
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import lesica as L

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)
started = time.time()
assets = {a["path"]: a for a in L.assets()}
print(f"dandiset {L.DANDISET} version {L.VERSION}: {len(assets)} assets")
lib = L.open_asset(assets["sub-Expt1/sub-Expt1_ses-StimulusLibrary_image.nwb"]["asset_id"])
T = L.load_templates(lib, ("NAT", "WN"))
clips = {k: lib["stimulus/templates"][k]["data"][()] for k in ["clip_nr_b1", "clip_nr_b2", "clip_nr_b3",
                                                               "clip_wr_b1", "clip_wr_b2", "clip_wr_b3"]}
lib_rates = {k: float(lib["stimulus/templates"][k]["starting_time"].attrs["rate"]) for k in ["NAT", "WN"]}
lib.close()
print(f"stimulus library read in {time.time() - started:.0f} s")

## Data check

Every recording file is opened and its trials table, units and stimulus series are read (the
files' full contents were listed while preparing; the columns used are named in `lesica.py`).
Each check of the metadata against the values is stated under its output.

In [ ]:
files = sorted(p for p in assets if "ecephys" in p)
inv = []
for p in files:
    f = L.open_asset(assets[p]["asset_id"])
    s = L.load_session(f)
    tr = s["trials"]
    for i in range(len(tr["trial_key"])):
        if tr["modality"][i] in ("NAT", "SBN") and not tr["derived"][i]:
            fr = tr["frame_times"][i]
            inv.append({"file": p.split("/")[1].replace("_ecephys.nwb", ""), "subject": s["subject"],
                        "units": len(s["units"]["unit_name"]), "code": tr["code"][i], "regime": tr["regime"][i],
                        "contrast": tr["contrast"][i], "rms": tr["contrast_rms"][i],
                        "template": tr["stim_template"][i], "frames": f'{tr["stim_start"][i]}-{tr["stim_stop"][i]}',
                        "seconds": round(float(tr["duration_s"][i])), "n_reps": tr["n_reps"][i],
                        "frame_hz": round(1 / np.median(np.diff(fr[0])), 3) if fr is not None else np.nan,
                        "frames_in_order": bool(np.all(np.diff(fr[1]) == 1)) if fr is not None else None})
    f.close()
inv = pd.DataFrame(inv)
print("Units per file and subject:")
print(inv.groupby(["subject", "file"]).units.first().to_string())
print("\nNatural movie (NAT) and binary noise (SBN) recordings, by subject:")
print(inv.groupby(["subject", "regime", "template", "contrast", "rms", "seconds", "n_reps"], dropna=False).size().to_string())
# Per file: the unique (non-repeat) movie and noise recordings, and which of them the analysis uses
# (L._trial picks the first matching row of the trials table)
used = []
for p in files:
    if not p.startswith(("sub-Expt1/", "sub-Expt2/")):
        continue
    f = L.open_asset(assets[p]["asset_id"])
    s = L.load_session(f)
    tr = s["trials"]
    pick = {L._trial(tr, "ns", "H"), L._trial(tr, "ns", "L"), L._trial(tr, "wn", "H", modality="SBN"), L._trial(tr, "wn", "L", modality="SBN")}
    for i in range(len(tr["trial_key"])):
        if tr["regime"][i] == "unique" and tr["modality"][i] in ("NAT", "SBN") and not tr["derived"][i]:
            used.append({"file": p.split("/")[1].replace("_ecephys.nwb", ""), "trial_key": tr["trial_key"][i], "code": tr["code"][i],
                         "template": tr["stim_template"][i], "contrast": tr["contrast"][i], "seconds": round(float(tr["duration_s"][i])),
                         "used": i in pick and (tr["stim_template"][i] == "NAT" or (p.startswith("sub-Expt1/") and s["session"].endswith(("a11", "a13", "a14"))))})
    f.close()
used = pd.DataFrame(used)
print("\nUnique movie and noise recordings of Expt1 and Expt2, and which the analysis uses:")
print(used.to_string(index=False))

Expt1 holds, in penetrations a11, a13 and a14, the 15 min movie at RMS contrast 0.40 and
0.15, the long binary noise at 0.55 and 0.20, and 120 repetitions of three 250-frame clips of
each stimulus at each contrast (12.5 s in all: the paper's "120 repeated trials of different
12 s segments"). Penetrations b11, b13 and b14 hold the 15 min movie (b11 also the noise, b14
only its high contrast), without repeats. Expt2 holds, in all eight penetrations, the 15 min movie
at both contrasts and 24 repetitions of a 90 s movie segment at both contrasts (the paper's 26-cell
cross-validation set). Expt3 and Expt4 hold no 15 min movie at two contrasts: they are the data of
the other three papers the dandiset supports. So the paper's data are in Expt1 and Expt2, two
animals. The contrasts match the paper's 0.40, 0.15, 0.55 and 0.20.

The per-file table shows which recordings are used. Expt1 a12 and b12 hold only a 100 s noise
test at high contrast ("wntest", which the analysis never selects), so they have no movie and are
not used. In a11, a13 and a14 the long noise used is the 600 s HC and the 901 s LC recording, not
the 100 s test. b11 also has the long noise at both contrasts but no 12 s repeats, so it is not
in the paper's 31-cell white-noise subset ("a 6 min segment of ... white noise ... along with 120
repeated trials"); its noise is not analysed.

Differences from the paper's description, from the table: the long noise lasts 600 s (high
contrast) and 900 s (low contrast), where the paper says 6 min; the first 6 min are used. Frames
are shown at 59.96 Hz (the paper: 120 Hz refresh, "pixel intensities updated every other
refresh", which is 60 Hz; its "50 Hz" is the movie's interpolated rate).

In [ ]:
nat = T["NAT"].reshape(len(T["NAT"]), -1)
fs = nat.std(1)
print("NAT template:", T["NAT"].shape, "frame SD percentiles 0/1/50/99/100:", np.round(np.percentile(fs, [0, 1, 50, 99, 100]), 3),
      "; frame means range", np.round([nat.mean(1).min(), nat.mean(1).max()], 3), "; frames with SD < 0.5:", int((fs < 0.5).sum()))
print("WN template values:", np.unique(T["WN"][:200]), "; library frame rates:", lib_rates)
same = [k for k in clips if any(np.allclose(clips[k][0], T["NAT"][j]) for j in range(0, len(T["NAT"]), 1))] if False else []
print("Clip shapes:", {k: v.shape for k, v in clips.items()})
print("\nRMS contrast of a frame is SD/mean; with a mean of 125 gray levels, a movie frame at 0.40 has SD 50 gray levels.")
print("Stimuli are used as 125 x rms x template (gray levels about the mean).")

The movie template holds frames of zero mean and unit SD (the library says "mean-subtracted";
each frame is rescaled, as the paper says "each frame was rescaled to have an RMS contrast"). A
few frames are blank, listed above. The noise template is +-1.

In [ ]:
# Clock: the 90 s repeats of Expt2 are stated to show movie frames 0-5400, the first 90 s of the
# 15 min movie. If that and the clocks hold, the repeat PSTH (one bin per frame) correlates with
# the response to the first 5400 frames of the unique movie at zero shift and not at others.
# Computed in the main pass below (clock_cc, shifts -6..6 frames); shown after it.
print("see the clock check after the main pass")

## Choice of the ridge penalty

The paper estimates RFs by recursive least squares with initial conditions g = 0 and K = D x I.
Its final estimate equals the batch ridge solution with penalty 1/D, which is computed here
directly (faster, same result). The paper gives D = 0.001 (Estimation of Receptive Fields,
paragraph 3) on a stimulus "scaled to have zero mean and possible values between -106 and 106"
(Natural Scene Movie and White Noise Stimuli, paragraph 1), which is gray levels about the mean
of 125, the scale used here: a penalty of 1000. It also says how D was chosen: "the value of D that
produced the RF estimates that provided the most accurate predictions of responses to natural
scene movies". This cell applies that rule on held-out movie data (the last 10% of the 15 min
movie, so as not to use the 90 s test segments), over a grid that includes the paper's literal
penalty and penalties relative to the stimulus variance from 1 to 30000 times it. The penalty is
set relative to the stimulus variance (an assumption: a fixed penalty in gray levels regularizes
the low-contrast RFs about seven times more strongly than the high-contrast ones). The cell also
prints, for each penalty, its ratio to the mean eigenvalue of X'X of one segment (the number of
frames in a segment times the stimulus variance): above 1, the movie estimate moves towards the
spike-triggered average. The literal penalty (`paper`) and 1000 x variance (`rel1000`, below the
mean eigenvalue) are carried through the whole analysis as variants (section "Ridge penalty
variants").

In [ ]:
cases, labels = [], []
for p, units in [("sub-Expt1/sub-Expt1_ses-a13_ecephys.nwb", [0, 1]), ("sub-Expt2/sub-Expt2_ses-a11_ecephys.nwb", [1, 4])]:
    f = L.open_asset(assets[p]["asset_id"])
    s = L.load_session(f)
    tr = s["trials"]
    for c in ("H", "L"):
        i = L._trial(tr, "ns", c)
        ft, idx = tr["frame_times"][i]
        F = L._frames(T["NAT"], idx)
        for u in units:
            r = L.frame_counts(s["units"]["spikes"][u], ft)
            r0, c0 = L.window(L.locate(L._frames(T["NAT"], tr["frame_times"][L._trial(tr, "ns", "H")][1]),
                                       L.frame_counts(s["units"]["spikes"][u], tr["frame_times"][L._trial(tr, "ns", "H")][0])))
            S = np.ascontiguousarray(F[:, r0:r0 + L.WIN, c0:c0 + L.WIN]).reshape(len(F), -1) * np.float32(125 * float(tr["contrast_rms"][i]))
            cases.append((S, r))
            labels.append(f"{s['subject']} {s['session']} {s['units']['unit_name'][u]} {c}")
    f.close()
specs = [("abs", 1000.0)] + [("rel", x) for x in (1.0, 10.0, 100.0, 1000.0, 3000.0, 10000.0, 30000.0)]
names = ["paper 1000"] + [f"{x:g} x var" for _, x in specs[1:]]
t0 = time.time()
cc = L.choose_ridge(cases, specs)
tab = pd.DataFrame(cc, index=labels, columns=names)
print("correlation of the held-out prediction with the frame counts (last 10% of the movie):")
print(tab.round(3).to_string())
print("mean:", tab.mean().round(4).to_dict())
nseg = [int(0.9 * len(r)) / L.NSEG for _, r in cases]
var = [float(np.mean(np.square(S, dtype=np.float64))) for S, _ in cases]
print("\npenalty over the mean eigenvalue of X'X of one segment (frames per segment x variance), range over the cases:")
for (k, v), n in zip(specs, names):
    ratio = [L.penalty(k, v, va) / (ns * va) for va, ns in zip(var, nseg)]
    print(f"  {n}: {min(ratio):.2g} to {max(ratio):.2g}")
print("stimulus variance (gray levels squared), HC and LC:", sorted({round(v) for v in var}))
del cases
best = int(np.argmax(cc.mean(0)))
assert specs[best][0] == "rel"
L.RIDGE_REL = specs[best][1]
print(f"chosen penalty: {L.RIDGE_REL:g} x stimulus variance ({time.time() - t0:.0f} s)")

The held-out prediction is best at 10000 x variance (mean r = 0.437; 0.434 at 3000 and 0.420 at
1000 x variance), which is chosen, and worst at the paper's literal penalty (0.170) and 1 x variance
(0.172). The literal penalty is 0.00008 to 0.0006 of the mean eigenvalue of X'X of a segment, so
the estimate is close to unregularized least squares, and the 360 s of noise give about 2400 frames
per segment for its 5292 parameters. The chosen penalty is 1.9 times the mean
eigenvalue.

## Main pass

For every unit of the eleven penetrations with the 15 min movie at both contrasts (Expt1 a11, a13,
a14, b11, b13, b14; Expt2 a11, a12, b11, c11, d11, e21, f31, g31): spike counts per frame, the RF
window (21 x 21 pixels centred on the peak of the spike-triggered average of the high-contrast
movie), RFs from the high- and low-contrast movie with the same number of spikes, each the mean of
nine segment estimates, the RF from shuffled responses under the five seeds of
`sample-composition`, and the NL, each under the chosen penalty and the two variants (`rel1000`, `paper`); the RF at
the paper's resolution, 128 Hz bins and 24 lags of 8 ms, in a 15 x 15 window centred on the
maximal point of the HC movie RF (`lesica.rf8`; used for the temporal width, latency and biphasic
ratio); for the Expt1 penetrations with repeats (a11, a13, a14) the same
for the first 6 min of the noise; and the repeat responses. In Expt2 the first 5400 movie frames,
which the 90 s repeats show again, are left out of the RF and NL estimates. Penetrations run in
parallel, one BLAS thread each.

In [ ]:
seeds = [L.seed("sample-composition") + k for k in (0, 1000, 2000, 3000, 4000)]
jobs = []
for e, sess in L.SESSIONS.items():
    for j, p in enumerate(sess):
        path = f"sub-{e}/sub-{e}_ses-{p}_ecephys.nwb"
        kw = dict(ns_skip=5400 if e == "Expt2" else 0, wn=(e == "Expt1" and p in ("a11", "a13", "a14")), key=(int(e[-1]), j))
        jobs.append((path, assets[path]["asset_id"], seeds, kw))
jobs.sort(key=lambda j: -(2 * j[3]["wn"] + 1))
t0 = time.time()
results = L.run_all(T, jobs)
print(f"main pass: {time.time() - t0:.0f} s")

In [ ]:
units = {(d["subject"], d["session"], d["unit_name"]): d for _, _, out in results for d in out}
df = pd.DataFrame(L.unit_table(results))
dfv = {w: pd.DataFrame(L.unit_table(results, which=w)) for w in L.PENALTIES if w != "main"}
print("units:", len(df), "; units with an error:", int(df.error.notna().sum()), df.error.dropna().tolist())
for c in L.CONDITIONS:
    n = df[c + "_snr"].notna().sum() if c + "_snr" in df else 0
    n8 = df[c + "_temporal_width_8"].notna().sum() if c + "_temporal_width_8" in df else 0
    print(f"{c}: RFs estimated for {n} units (128 Hz RFs: {n8}); seconds used {df[c + '_seconds'].min():.0f}-{df[c + '_seconds'].max():.0f}; "
          f"spikes {df[c + '_n_spikes'].min():.0f}-{df[c + '_n_spikes'].max():.0f}")
print("\nHC and LC spike counts used are equal by construction; largest difference:",
      float((df.ns_H_n_spikes - df.ns_L_n_spikes).abs().max()), "spikes")
lam = pd.DataFrame([{"cond": c, "penalty": w, "lam": (units[k][c] if w == "main" else units[k][c]["variants"][w])["lam"]}
                    for k in units for c in L.CONDITIONS if c in units[k] and "rf" in units[k][c] for w in L.PENALTIES])
print("\npenalties used (gray levels squared), by condition:")
print(lam.groupby(["penalty", "cond"]).lam.agg(["min", "max"]).round(0).to_string())
cl = np.array([units[k]["clock_cc"] for k in units if "clock_cc" in units[k]])
print("\nClock check, Expt2 (90 s repeats against movie frames 0-5400), median correlation over", len(cl), "units by shift (frames):")
print(dict(zip(range(-6, 7), np.round(np.nanmedian(cl, 0), 3))))
print("units whose best shift is 0:", int((np.nanargmax(cl, 1) == 6).sum()), "of", len(cl))

The clock check shows the 90 s repeats are the first 90 s of the 15 min movie (correlation
peaks at zero shift); the paper calls them "a different 90 s movie segment". For Expt1, which has
no shared segment, the RFs themselves are the check: a peak at 2 to 3 frames (33 to 50 ms) after
the stimulus, and at 30 to 50 ms in the 128 Hz RFs, shown below, would not arise with a broken
clock.

In [ ]:
lat = df[["ns_H_latency", "ns_L_latency", "wn_H_latency", "wn_L_latency", "ns_H_latency_8", "ns_L_latency_8", "wn_H_latency_8", "wn_L_latency_8"]] * 1000
print("RF peak latency (ms), frame-rate RFs and 128 Hz RFs (_8), by subject:")
print(df.assign(**{c: lat[c] for c in lat}).groupby("subject")[list(lat)].describe().round(1).T.to_string())

## sample-composition

> The 15 min movie analyses include 68 LGN cells whose RF peak is at least ten times the noise, 44
> ON-center (19 X, 19 Y, 6 unclassified) and 24 OFF-center (11 X, 8 Y, 5 unclassified); 26 of them
> have the 90 s repeats and 31 have white noise and the 12 s repeats.

Inclusion: the largest absolute value of the HC and of the LC movie RF over the SD of the RF from
shuffled responses, at least 10; readings (a) both contrasts, (b) either. ON or OFF is the sign of
the maximal point of the HC movie RF (the files carry no ON/OFF label); X or Y is the units table's
`xy_class`. The downstream claims use reading (a). Units whose RFs change sign between
conditions at their maximal point are counted (`sign_ok` is false for them); they are kept in the
main analysis, as the paper's rule does not exclude them, and left out in variants where ON/OFF
matters.

In [ ]:
inc_a = (df.ns_H_snr >= 10) & (df.ns_L_snr >= 10)
inc_b = (df.ns_H_snr >= 10) | (df.ns_L_snr >= 10)
print(f"Units with both 15 min movies: {len(df)}; passing at both contrasts (reading a): {inc_a.sum()}; at either (b): {inc_b.sum()}")
for k, sd in enumerate((0, 1000, 2000, 3000, 4000)):
    a = df.ns_H_snrs.apply(lambda x: x[k]) >= 10
    b = df.ns_L_snrs.apply(lambda x: x[k]) >= 10
    print(f"  seed + {sd}: both {int((a & b).sum())}, either {int((a | b).sum())}")
print("peak/noise percentiles, HC:", np.round(np.percentile(df.ns_H_snr, [5, 25, 50, 75, 95]), 1),
      " LC:", np.round(np.percentile(df.ns_L_snr, [5, 25, 50, 75, 95]), 1))
for thr in (10, 15, 20, 25):
    print(f"  threshold {thr}: both {int(((df.ns_H_snr >= thr) & (df.ns_L_snr >= thr)).sum())}")
df["onoff"] = np.where(df.ns_H_sign > 0, "ON", "OFF")
df["xy"] = df.xy_class.replace("", "unclassified").str.upper().replace("UNCLASSIFIED", "unclassified")
print("units whose HC and LC movie RFs have opposite signs at their maximal point:", int((df.ns_H_sign != df.ns_L_sign).sum()),
      "; of the included (reading a):", int((df.ns_H_sign != df.ns_L_sign)[inc_a].sum()))
sg = df[[c + "_sign" for c in L.CONDITIONS]]
df["sign_ok"] = sg.apply(lambda r: r.dropna().nunique() == 1, axis=1)
print("included units whose frame-rate RFs agree in sign in every condition they have:", int(df.sign_ok[inc_a].sum()), "of", int(inc_a.sum()))
print("included units whose 128 Hz HC movie RF has the sign of the frame-rate one:", int((df.ns_H_sign_8 == df.ns_H_sign)[inc_a].sum()))
l8 = df.loc[inc_a, ["ns_H_latency_8", "ns_L_latency_8"]] * 1000
print("included units with a 128 Hz movie latency below 16 ms or above 120 ms (HC, LC):", int(((l8.ns_H_latency_8 < 16) | (l8.ns_H_latency_8 > 120)).sum()),
      int(((l8.ns_L_latency_8 < 16) | (l8.ns_L_latency_8 > 120)).sum()))
for name, m in [("reading a (both)", inc_a), ("reading b (either)", inc_b)]:
    d = df[m]
    print(f"\n{name}: {len(d)} cells; by subject {d.subject.value_counts().to_dict()}")
    print(pd.crosstab(d.onoff, d.xy, margins=True).to_string())
    print("with the long noise and the 12 s repeats (Expt1 a11, a13, a14):", int(d.session.isin(["a11", "a13", "a14"]).where(d.subject == "Expt1", False).sum()),
          "; with the 90 s repeats (Expt2):", int((d.subject == "Expt2").sum()))
inc = df[inc_a].copy()
wn = inc[(inc.subject == "Expt1") & inc.session.isin(["a11", "a13", "a14"])].copy()
ex2 = inc[inc.subject == "Expt2"].copy()

In [ ]:
# Units recorded in more than one penetration: the same electrode in a11, a13 and a14 of Expt1
# ("electrode adjusted to isolate units better, same presumed eccentricity"), and Expt2 a11 and
# a12. Units on the same electrode are compared by the correlation of their HC movie RFs placed
# in screen coordinates.
def screen_rf(d):
    rf = np.zeros((L.NLAGS, 48, 48))
    r0, c0 = d["window"]
    rf[:, r0:r0 + L.WIN, c0:c0 + L.WIN] = d["ns_H"]["rf"].reshape(L.NLAGS, L.WIN, L.WIN)
    return rf.ravel()
pairs = []
keys = [k for k in units if "ns_H" in units[k] and "rf" in units[k]["ns_H"]]
for i, a in enumerate(keys):
    for b in keys[i + 1:]:
        if a[0] == b[0] and a[1] != b[1] and units[a]["electrode_number"] == units[b]["electrode_number"]:
            pairs.append({"a": " ".join(a), "b": " ".join(b), "r": np.corrcoef(screen_rf(units[a]), screen_rf(units[b]))[0, 1]})
pairs = pd.DataFrame(pairs)
print("pairs on the same electrode in different penetrations:", len(pairs), "; RF correlation > 0.8:", int((pairs.r > 0.8).sum()))
print(pairs[pairs.r > 0.8].round(3).to_string(index=False))
print("possible_duplicate_of entries in the units tables:", df.possible_duplicate_of.replace("", np.nan).dropna().tolist())

**Result: not reproduced.** Under the chosen penalty (10000 x stimulus variance), 111 of the 143
units pass the inclusion rule at both contrasts (133 at either), against the paper's 68, under all
five seeds of the shuffles (111 each; 132 to 134 at either). The subset with the long noise and the
12 s repeats is 31 cells, as in the paper; the subset with the 90 s repeats is 58, against 26. Among
the 111, 55 are ON and 56 OFF (paper 44 and 24), and 60 X, 43 Y and 8 unclassified (paper 30, 27 and
11). The count depends on the threshold (61 cells pass at 15) and on the penalty (section "Ridge
penalty variants"). 13 of the 111 have HC and LC movie RFs of opposite sign at their maximal point;
94 keep their sign in every condition. No pair of units on the same electrode in different
penetrations has RFs that correlate above 0.8, so no unit appears to have been counted twice.

## firing-rate-contrast and firing-rate-lc-ns-wn-similar

> Across the 31 cells with all four stimuli, mean firing rates are higher at high than at low
> contrast for both natural movies (10.5 against 6.5 Hz) and white noise (8.2 against 5.7 Hz).

> Mean firing rates to low-contrast natural movies and low-contrast white noise do not differ across
> the 31 cells.

The 31 cells are the included cells of Expt1 a11, a13 and a14, the only penetrations with the long
noise and the 12 s repeats. Readings P2: (a) the long segments (15 min movie, first 6 min of noise),
(b) the 12 s repeats.

In [ ]:
rep = {k: L.repeat_measures(units[(r.subject, r.session, r.unit_name)]) for k, r in wn.iterrows()}
for c in L.CONDITIONS:
    wn["rep_rate_" + c] = [rep[k][c]["rate"] for k in wn.index]
    wn["rel_a_" + c] = [rep[k][c]["rel_a"] for k in wn.index]
    wn["rel_b_" + c] = [rep[k][c]["rel_b"] for k in wn.index]
    wn["sparse_" + c] = [rep[k][c]["sparse"] for k in wn.index]
print("valid repetitions per block, range over cells:", min(min(rep[k][c]["n_reps"]) for k in wn.index for c in L.CONDITIONS),
      "-", max(max(rep[k][c]["n_reps"]) for k in wn.index for c in L.CONDITIONS))
def paired(a, b):
    t = stats.ttest_rel(a, b)
    return t.statistic, t.pvalue
rate_rows = []
for reading, cols in [("a: long segments", ["rate_ns_H", "rate_ns_L", "rate_wn_H", "rate_wn_L"]),
                      ("b: 12 s repeats", ["rep_rate_ns_H", "rep_rate_ns_L", "rep_rate_wn_H", "rep_rate_wn_L"])]:
    m = wn[cols].mean(); s = wn[cols].std()
    tns, pns = paired(wn[cols[0]], wn[cols[1]]); tw, pw = paired(wn[cols[2]], wn[cols[3]]); tc, pc = paired(wn[cols[1]], wn[cols[3]])
    rate_rows.append({"reading": reading, "n": len(wn), "HC NS": f"{m.iloc[0]:.2f} ({s.iloc[0]:.2f})", "LC NS": f"{m.iloc[1]:.2f} ({s.iloc[1]:.2f})",
                      "HC WN": f"{m.iloc[2]:.2f} ({s.iloc[2]:.2f})", "LC WN": f"{m.iloc[3]:.2f} ({s.iloc[3]:.2f})",
                      "p NS HC>LC": pns, "t NS": tns, "p WN HC>LC": pw, "t WN": tw, "p LC NS vs LC WN": pc, "LC NS - LC WN": m.iloc[1] - m.iloc[3]})
rates = pd.DataFrame(rate_rows)
print(rates.to_string(index=False))
print("\nby penetration (reading a):")
print(wn.groupby("session")[["rate_ns_H", "rate_ns_L", "rate_wn_H", "rate_wn_L"]].mean().round(2).to_string())
print("\n15 min movie rates over all", len(inc), "included cells, by subject:")
print(inc.groupby("subject")[["rate_ns_H", "rate_ns_L"]].agg(["mean", "std", "count"]).round(2).to_string())

**Result: partially reproduced.** Over the 31 cells and the long segments the four means are 8.68,
6.11, 6.54 and 5.35 Hz, all within the paper's SDs of 10.5, 6.5, 8.2 and 5.7 Hz. HC is higher than LC
for the movie (p = 5.0e-07) and for the noise (p = 0.0022), so the noise test misses the paper's
p < 0.001. Over the 12 s repeats the movie test gives p = 0.022.

In [ ]:
for _, r in rates.iterrows():
    print(f"{r.reading}: LC NS against LC WN, difference {r['LC NS - LC WN']:.2f} Hz, paired t p = {r['p LC NS vs LC WN']:.3g}; "
          f"p > 0.05: {r['p LC NS vs LC WN'] > 0.05}; p > 0.2: {r['p LC NS vs LC WN'] > 0.2}")

**Result: reproduced, under the long-segment reading (an assumption).** With rates over the long
segments the LC NS and LC WN rates differ by 0.75 Hz, paired t p = 0.063 (above 0.05, not above the
paper's 0.2). With rates over the 12 s repeats they differ by 1.37 Hz, p = 0.0013. The readings
disagree; the long segments are adopted, as the paper's comparison concerns the stimuli used to fit
the RFs and NLs.

## ln-prediction-accuracy

> LN models fitted from the 15 min movie predict responses to a novel 90 s movie segment with
> correlation coefficients of 0.70 at high and 0.76 at low contrast (26 cells, 8 ms bins).

Included cells of Expt2. The RF and the mean NL of each contrast (from movie frames 5400 to 54000,
so that the test frames 0-5400 are not in the training data) predict the 8 ms rate during the 24
repeats. Readings: (a) correlation with the trial-averaged response, (b) mean over trials of the
correlation with each trial.

In [ ]:
def predict(d, c):
    """LN prediction of the 90 s repeats: the unit's RF and mean NL (nine segments) for contrast c,
    applied to movie frames 0-5400 as shown in the repeats; 8 ms bins."""
    r0, c0 = d["window"]
    a, b = d["rep90_frames"]
    F = T["NAT"][a:b, r0:r0 + L.WIN, c0:c0 + L.WIN].reshape(b - a, -1) * np.float32(125 * d["rms"]["ns_" + c])
    y = L.filtered(F, d["ns_" + c]["rf"])
    R = d["rep90_" + c].astype(float)
    tc = (np.arange(R.shape[1]) + 0.5) * L.BIN
    fr = np.floor(tc / L.REP_DT).astype(int)
    ok = (fr >= L.NLAGS - 1) & (fr < len(y))
    pred = np.interp(y[fr[ok]], d["ns_" + c]["nl_x"], d["ns_" + c]["nl_y"])
    Rk = R[:, ok]
    ca = np.corrcoef(pred, Rk.mean(0))[0, 1]
    cb = np.nanmean([np.corrcoef(pred, x)[0, 1] for x in Rk if x.std() > 0])
    return ca, cb, len(R)
pr = []
for k, r in ex2.iterrows():
    d = units[(r.subject, r.session, r.unit_name)]
    row = {"session": r.session, "unit": r.unit_name, "xy": r.xy}
    for c in ("H", "L"):
        row[f"a_{c}"], row[f"b_{c}"], row[f"reps_{c}"] = predict(d, c)
    pr.append(row)
pr = pd.DataFrame(pr)
print(f"{len(pr)} cells; repetitions per contrast {pr.reps_H.min()}-{pr.reps_H.max()}")
for rd in ("a", "b"):
    print(f"reading {rd}: HC {pr[f'{rd}_H'].mean():.3f} (SD {pr[f'{rd}_H'].std():.3f}), LC {pr[f'{rd}_L'].mean():.3f} (SD {pr[f'{rd}_L'].std():.3f})")
print("\nby penetration (reading a):")
print(pr.groupby("session")[["a_H", "a_L"]].agg(["mean", "count"]).round(3).to_string())

**Result: not reproduced.** Over the 58 included Expt2 cells, the LN predictions correlate with the
trial-averaged 8 ms response at 0.477 (HC) and 0.532 (LC), below the paper's 0.70 and 0.76 (tolerance
0.07); with single trials, 0.282 and 0.259. LC is higher than HC under reading (a), as in the paper.

## Receptive field properties (Figure 3)

> fig3b-spatial-width-contrast: A decrease in contrast has no significant effect on the width of the
> spatial RF center, for natural movies (68 cells) or white noise (31 cells).

> fig3b-spatial-width-correlations: A decrease in correlations (low-contrast natural movie to
> low-contrast white noise) decreases the width of the spatial RF center by 20% on average (31 cells).

> fig3c-surround-contrast: A decrease in contrast decreases the relative strength of the RF surround
> by 17% for natural movies and 18% for white noise.

> fig3c-surround-correlations: A decrease in correlations decreases the relative strength of the RF
> surround by 34%.

> fig3d-temporal-width-contrast: A decrease in contrast widens the temporal profile of the RF center
> (slower dynamics) by 16% for natural movies and 15% for white noise.

> fig3d-temporal-width-correlations: A decrease in correlations widens the temporal profile of the RF
> center by 15%.

Each table gives, per comparison, the means, `pct_a` (P1 a, change of the means), `pct_b` (P1 b,
mean of per-cell changes), `ratio_c` (ratio of the means, %), and the paired t test. The NS HC to
LC comparison is shown over all included cells (the claim's 68) and over the 31 Expt1 cells.
Surround/center ratio readings: `_a` the surround's largest absolute value, `_b` its extreme of the
sign opposite to the centre. Temporal width, latency and biphasic ratio come from the 128 Hz RFs
with 24 lags of 8 ms (suffix `_8`, the paper's resolution and P6's 8 ms grid); the frame-rate
values (no suffix, 16.7 ms lags) are a variant.

In [ ]:
def compare(prop, a, b, data):
    x, y = data[a + "_" + prop].astype(float), data[b + "_" + prop].astype(float)
    group = f"all {len(inc)}" if len(data) == len(inc) else f"{len(data)} (Expt1)"
    ok = x.notna() & y.notna()
    t = stats.ttest_rel(x[ok], y[ok])
    pc = L.pct_change(x[ok], y[ok])
    return {"prop": prop, "comparison": f"{a}->{b}", "n": int(ok.sum()), "mean_a": x[ok].mean(), "mean_b": y[ok].mean(),
            "sd_a": x[ok].std(), "sd_b": y[ok].std(), "pct_a": pc["a"], "pct_b": pc["b"], "ratio_c": pc["c"], "t": t.statistic, "p": t.pvalue, "group": group}
rows = []
for prop in L.PROPS:
    rows.append(compare(prop, "ns_H", "ns_L", inc))
    rows.append(compare(prop, "ns_H", "ns_L", wn))
    rows.append(compare(prop, "wn_H", "wn_L", wn))
    rows.append(compare(prop, "ns_L", "wn_L", wn))
    rows.append(compare(prop, "ns_L", "wn_H", wn))
cmp = pd.DataFrame(rows)
def show(props):
    print(cmp[cmp.prop.isin(props)].to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("Units are seconds for widths and latency (temporal), degrees for spatial width; gain in Hz per filtered unit.")
print("Spatial widths with failed DoG fits (NaN):", {c: int(inc[c + "_spatial_width"].isna().sum()) for c in ["ns_H", "ns_L"]},
      {c: int(wn[c + "_spatial_width"].isna().sum()) for c in ["wn_H", "wn_L"]})
show(["spatial_width"])

**Result: reproduced.** The spatial RF width does not change with contrast: 0.688 against 0.704 deg
for the movie (110 cells with both fits, p = 0.15) and 0.548 against 0.563 deg for the noise (31
cells, p = 0.46).

In [ ]:
show(["spatial_width"])
print("\nby subject, NS HC->LC:")
print(inc.groupby("subject")[["ns_H_spatial_width", "ns_L_spatial_width"]].mean().round(3).to_string())

**Result: reproduced.** From LC NS to LC WN the width falls from 0.694 to 0.563 deg, a decrease of
18.9% (P1 a) or 20.5% (P1 b), within 18 to 22%, paired t p = 3.9e-07 over 31 cells.

The verified outcome is kept, and a change to not reproduced is proposed in the attempt record.
Under 1000 x variance the decrease is 5.1% (p = 0.11; section "Ridge penalty variants"), and
held-out prediction barely separates the two penalties (mean r 0.420 against 0.437, section "Choice
of the ridge penalty"). The round-2 verifier reports that in its own synthetic check this estimator,
under the chosen penalty, gives a narrower noise-driven RF centre from the same true RF; that check
is the verifier's and is not run here.

In [ ]:
show(["surround_ratio_a", "surround_ratio_b"])
print("\nby subject, NS:")
print(inc.groupby("subject")[["ns_H_surround_ratio_a", "ns_L_surround_ratio_a", "ns_H_surround_ratio_b", "ns_L_surround_ratio_b"]].mean().round(4).to_string())
print("centre sizes (pixels), median by condition:", {c: float(inc[c + "_n_center"].median()) for c in ["ns_H", "ns_L"]},
      {c: float(wn[c + "_n_center"].median()) for c in ["wn_H", "wn_L"]})

**Result: not reproduced.** For the movie, the surround/center ratio falls with contrast, by 11.6%
(P1 a) or 6.7% (P1 b) over 111 cells (p = 2.5e-05; 16.8% over the 31 Expt1 cells, p = 0.00073), below
the paper's 17%. For the noise it does not change (0.0595 against 0.0591, p = 0.93), against the
paper's 18%, so the white-noise test fails. Reading (b) of the surround peak gives the same.

In [ ]:
show(["surround_ratio_a", "surround_ratio_b"])

**Result: reproduced.** From LC NS to LC WN the surround/center ratio falls from 0.0964 to 0.0591,
a decrease of 38.7% (P1 a) or 37.2% (P1 b); 37.2% is within 30.6 to 37.4%. Paired t p = 2.7e-10 over
31 cells. With the opposite-sign surround peak (reading b) the decreases are 44.2% and 43.1%.

The verified outcome is kept, and a change to not reproduced is proposed in the attempt record:
under 1000 x variance the decrease is 2.7% (p = 0.67, reading a; section "Ridge penalty variants"),
and the round-2 verifier reports that in its synthetic check the chosen penalty gives a weaker
noise-driven surround from the same true RF (not run here).

In [ ]:
show(["temporal_width_8", "temporal_width"])
print("\nby subject, NS (ms):")
print((inc.groupby("subject")[["ns_H_temporal_width_8", "ns_L_temporal_width_8", "ns_H_temporal_width", "ns_L_temporal_width"]].mean() * 1000).round(1).to_string())
print("128 Hz temporal width range (ms):", {c: np.round(np.array([inc[c + '_temporal_width_8'].min(), inc[c + '_temporal_width_8'].max()]) * 1000, 1).tolist() for c in ["ns_H", "ns_L"]},
      {c: np.round(np.array([wn[c + '_temporal_width_8'].min(), wn[c + '_temporal_width_8'].max()]) * 1000, 1).tolist() for c in ["wn_H", "wn_L"]})

**Result: partially reproduced.** With the 128 Hz RFs (24 lags of 8 ms), the temporal width widens
at low contrast for the movie, from 24.1 to 27.6 ms over 111 cells, by 14.4% (P1 a) or 16.8% (P1 b),
p = 1.4e-10; 16.8% is within 14.4 to 17.6%. For the noise it widens by 23.8% (P1 a) or 27.1% (P1 b)
over 31 cells, above 13.5 to 16.5%, with p = 0.011, which misses p < 0.001. With the frame-rate RFs
(variant) the movie width does not change (+2.4%, p = 0.36) and the noise width rises by 43%
(p = 0.0030).

The criterion's partial outcome requires the tests to hold and the sizes not; here the noise test
misses p < 0.001 (p = 0.011), so by the criterion the result is not reproduced. The verified outcome
is kept, and a change to not reproduced is proposed in the attempt record.

In [ ]:
show(["temporal_width_8", "temporal_width"])

**Result: not reproduced.** With the 128 Hz RFs the width rises from LC NS to LC WN by 16.1% (P1 a,
within 13.5 to 16.5%) or 18.4% (P1 b), but the paired t test gives p = 0.11 over 31 cells. With the
frame-rate RFs (variant) the rise is 17.6% or 23.3%, p = 0.057.

## surround-temporal-correlation

> Across cells and the HC and LC natural movie conditions, the relative surround strength and the
> temporal width of the RF center are negatively correlated (r = -0.5, p < 0.001).

Main: surround/center ratio of the frame-rate RF (21 x 21 window) and temporal width of the 128 Hz
RF. Variants: both from the frame-rate RF; both from the 128 Hz RF.

In [ ]:
for name, tw, sfx in [("main: 128 Hz width, frame-rate surround", "temporal_width_8", ""),
                      ("variant: both frame-rate", "temporal_width", ""), ("variant: both 128 Hz", "temporal_width_8", "_8")]:
    for rd in ("a", "b"):
        x = np.r_[inc[f"ns_H_surround_ratio_{rd}{sfx}"], inc[f"ns_L_surround_ratio_{rd}{sfx}"]].astype(float)
        y = np.r_[inc[f"ns_H_{tw}"], inc[f"ns_L_{tw}"]].astype(float)
        ok = np.isfinite(x) & np.isfinite(y)
        r, p = stats.pearsonr(x[ok], y[ok])
        by = {sub: round(stats.pearsonr(x[m], y[m])[0], 3) for sub in ("Expt1", "Expt2")
              for m in [np.r_[inc.subject == sub, inc.subject == sub] & ok]}
        print(f"{name}, surround reading {rd}: r = {r:.3f}, p = {p:.3g}, n = {ok.sum()} points ({len(inc)} cells x 2); by subject {by}")

**Result: partially reproduced.** With the frame-rate surround/center ratio and the 128 Hz temporal
width (main), r = -0.366 (p = 1.9e-08, surround reading a) and r = -0.331 (p = 4.4e-07, reading b)
over 222 points of 111 cells: negative and significant, but weaker than -0.5 (tolerance -0.55 to
-0.45). With both measures from the 128 Hz RFs, r = -0.466 (a) and -0.480 (b), p below 1e-12,
within tolerance; with both from the frame-rate RFs, r = -0.091 (p = 0.18) and -0.145 (p = 0.031).

The criterion has no partial outcome. The paper takes both measures from the same RF, so the reading
with both from the 128 Hz RFs is adopted as the stated assumption; under it r is within tolerance
(-0.466 and -0.480, p below 1e-12). The verified outcome is kept, and a change to reproduced, resting
on that assumption, is proposed in the attempt record. The penalty of the 128 Hz RFs was not varied
(see the summary).

## Nonlinearity properties (Figure 4)

> fig4b-gain-contrast: A decrease in contrast increases the gain of the nonlinearity by 89% for
> natural movies and 148% for white noise. fig4b-gain-correlations: a decrease in correlations
> increases the gain by 296%.

> fig4c-offset-contrast: A decrease in contrast decreases the (absolute) offset of the nonlinearity by
> 56% for natural movies and 59% for white noise. fig4c-offset-correlations-none: a decrease in
> correlations has no effect on the offset.

> fig4d-normalized-offset-contrast: A decrease in contrast increases the normalized offset by 14% for
> natural movies and 24% for white noise. fig4d-normalized-offset-correlations: a decrease in
> correlations increases it by 90%.

In [ ]:
# Offset adjusted per animal (P5): within each condition, each animal's mean q is subtracted and
# the grand mean added, within the set of cells compared. The 31 white-noise cells come from one
# animal (Expt1), so within that set the adjustment changes no condition; the NS contrast
# comparison over all included cells pools two animals and is adjusted. Normalized offset is the
# adjusted q over the SD of the filtered stimulus.
def adjust(data, cols):
    out = data.copy()
    for c in cols:
        g = out.groupby("subject")[c + "_offset"].transform("mean")
        out[c + "_offset_adj"] = out[c + "_offset"] - g + out[c + "_offset"].mean()
        out[c + "_norm_offset_adj"] = out[c + "_offset_adj"] / out[c + "_ysd"]
    return out
inc = adjust(inc, ["ns_H", "ns_L"])
wn = adjust(wn, L.CONDITIONS)
print("largest change made by the adjustment within the 31 cells:", float(max((wn[c + "_offset_adj"] - wn[c + "_offset"]).abs().max() for c in L.CONDITIONS)))
rows = []
for prop in ["gain", "offset_adj", "offset", "norm_offset_adj", "norm_offset"]:
    rows.append(compare(prop, "ns_H", "ns_L", inc))
    rows.append(compare(prop, "ns_H", "ns_L", wn))
    rows.append(compare(prop, "wn_H", "wn_L", wn))
    rows.append(compare(prop, "ns_L", "wn_L", wn))
    rows.append(compare(prop, "ns_L", "wn_H", wn))
nlc = pd.DataFrame(rows)
cmp = pd.concat([cmp[~cmp.prop.isin(nlc.prop)], nlc], ignore_index=True)
# Variant: the first round's uneven adjustment, the 31 cells' NS offsets taken from the two-animal
# adjustment and the WN offsets left as they are
wn_uneven = wn.copy()
for c in ["ns_H", "ns_L"]:
    wn_uneven[c + "_offset_adj"] = inc.loc[wn.index, c + "_offset_adj"]
    wn_uneven[c + "_norm_offset_adj"] = inc.loc[wn.index, c + "_norm_offset_adj"]
uneven = pd.DataFrame([compare(prop, "ns_L", b, wn_uneven) for prop in ["offset_adj", "norm_offset_adj"] for b in ["wn_L", "wn_H"]])
print("NL fits that failed (NaN gain):", {c: int(inc[c + "_gain"].isna().sum()) for c in ["ns_H", "ns_L"]}, {c: int(wn[c + "_gain"].isna().sum()) for c in ["wn_H", "wn_L"]})
print("offsets q at or below the lowest NL point (no zero part):", {c: int((inc[c + "_offset"] <= inc[c + "_offset"].min()).sum()) for c in ["ns_H", "ns_L"]})
show(["gain"])
print("\nby subject, NS gain:")
print(inc.groupby("subject")[["ns_H_gain", "ns_L_gain"]].mean().round(5).to_string())

**Result: partially reproduced.** Gain rises at low contrast for the movie (p = 3.6e-42, 111 cells)
and the noise (p = 2.4e-10, 31 cells). The noise increase, 153.9% (P1 b), is within 133 to 163%; the
movie increase is 124.7% (P1 a) or 144% (P1 b), above the paper's 89% (80 to 98%).

In [ ]:
show(["gain"])

**Result: partially reproduced.** Gain rises from LC NS to LC WN (p = 3.5e-06), by 60.5% (P1 a) or
66.4% (P1 b), and the ratio of the means is 160.5%, all far below the paper's 296%.

In [ ]:
show(["offset_adj", "offset"])
print("\nthe adjusted and unadjusted values are the same within the 31 Expt1 cells (one animal)")
print("\nby subject, NS offset (unadjusted):")
print(inc.groupby("subject")[["ns_H_offset", "ns_L_offset"]].mean().round(1).to_string())

**Result: reproduced.** The offset adjusted per animal falls at low contrast by 52.9% for the movie
(P1 a; 111 cells, p = 2.6e-13) and by 64.4% (P1 a) or 57.2% (P1 b) for the noise (31 cells,
p = 7.2e-10); 52.9% and 57.2% are within the paper's 56 and 59% (tolerance 5.6 and 5.9).
Unadjusted, the movie decrease is also 52.9% over 111 cells.

In [ ]:
show(["offset_adj"])
print("\nvariant, the first round's uneven adjustment (31 cells' NS offsets shifted by the two-animal adjustment, WN not):")
print(uneven.to_string(index=False, float_format=lambda v: f"{v:.4g}"))

**Result: not reproduced.** The 31 cells all come from Expt1, so the per-animal adjustment changes
none of their offsets (largest change 9e-13). The offset falls from LC NS to LC WN, 1956 against
1360 (-30.5%, P1 a), paired t p = 0.012, below 0.05. With the first round's uneven adjustment (the
movie offsets shifted by the two-animal adjustment, the noise offsets not) the decrease is 18.9% and
p = 0.16.

In [ ]:
show(["norm_offset_adj", "norm_offset"])
print("\nby subject, NS normalized offset (adjusted):")
print(inc.groupby("subject")[["ns_H_norm_offset_adj", "ns_L_norm_offset_adj"]].mean().round(3).to_string())

**Result: not reproduced.** The normalized offset rises at low contrast for the movie, by 42.3% (P1 a,
111 cells, p = 1.4e-05; the paper 14%), and does not change for the noise (0.676 against 0.676,
p = 0.99; the paper 24%).

In [ ]:
show(["norm_offset_adj"])

**Result: not reproduced.** From LC NS to LC WN the normalized offset rises by 27.6% (P1 a) or falls
by 8.1% (P1 b), against 90%, with paired t p = 0.12. With the first round's uneven adjustment
(printed under fig4c-offset-correlations-none) the rise is 50.1% (P1 a), p = 0.020.

## figs1-property-correlations

> Changes in RF and NL properties are correlated with each other across stimulus conditions; every
> pairwise correlation of the properties that adapt significantly has p < 0.001, except
> surround/center ratio against temporal width for contrast during white noise (p = 0.06).

Readings: (a) the four properties that adapt in all three comparisons; (b) every property the paper
reports as changing significantly in that comparison (offset for the contrast comparisons, spatial
width for correlations). Temporal width from the 128 Hz RFs.

In [ ]:
def pair_corrs(cols, conds, data):
    out = []
    for i, a in enumerate(cols):
        for b in cols[i + 1:]:
            x = np.r_[data[conds[0] + "_" + a], data[conds[1] + "_" + a]].astype(float)
            y = np.r_[data[conds[0] + "_" + b], data[conds[1] + "_" + b]].astype(float)
            ok = np.isfinite(x) & np.isfinite(y)
            r, p = stats.pearsonr(x[ok], y[ok])
            out.append({"pair": f"{a} x {b}", "r": r, "p": p, "n": int(ok.sum())})
    return pd.DataFrame(out)
four = ["surround_ratio_a", "temporal_width_8", "gain", "norm_offset_adj"]
comps = {"contrast NS": (["ns_H", "ns_L"], inc, four + ["offset_adj"]),
         "contrast WN": (["wn_H", "wn_L"], wn, four + ["offset_adj"]),
         "correlations": (["ns_L", "wn_L"], wn, four + ["spatial_width"])}
s1 = []
for rd in ("a", "b"):
    for name, (conds, data, props) in comps.items():
        t = pair_corrs(four if rd == "a" else props, conds, data)
        t["comparison"], t["reading"] = name, rd
        s1.append(t)
s1 = pd.concat(s1, ignore_index=True)
s1["named"] = (s1.comparison == "contrast WN") & (s1.pair == "surround_ratio_a x temporal_width_8")
print(s1.round(4).to_string(index=False))
for rd in ("a", "b"):
    t = s1[s1.reading == rd]
    ok = ((t.p < 0.001) & ~t.named) | ((t.p > 0.001) & t.named)
    print(f"reading {rd}: {int(ok.sum())} of {len(t)} pairs as the legend states ({ok.mean():.0%}); named pair p = {t[t.named].p.iloc[0]:.3g}")

**Result: not reproduced.** Only 2 of 18 pairs (reading a) and 6 of 30 (reading b) behave as the
legend states; most pairs have p > 0.001. The named pair (surround/center ratio and temporal width,
contrast in WN) has p = 0.23.

## fig5-within-trial-contrast

> Within a single presentation of the high-contrast movie, the contrast inside a cell's RF varies
> over time (0.27 to 0.43 over a 150 s segment for an example OFF-center Y cell), and RFs and NLs
> estimated from the periods in the top and bottom thirds of all its values show the same changes
> in surround, temporal dynamics, gain and offset as the HC and LC comparison.

The example cell cannot be identified, so every included OFF-center Y cell is tested (the
criterion asks for at least one). The share of cells with all four directions is reported over
these cells only, not over all included cells, to bound the run time. Contrast within the RF
(`lesica.rf_contrast`), luminance as displayed (125 x (1 + 0.40 x template), clipped to 0-255):
reading (a) per frame, the RMS contrast over the RF centre's pixels (unweighted); reading (b) per
pixel, the RMS contrast over a 1 s sliding window (an assumed length), averaged over the pixels
weighted by the RF's absolute spatial profile at its peak lag. The other two combinations of the
criterion (a weighted, b unweighted) are not run. The thirds are taken over the frames used for
the HC RF, by the contrast of the frame at the RF's peak lag before each response frame; each
third's RF (frame rate, nine groups of frames, the chosen penalty) and NL (8 ms bins in those
frames) are estimated as for the cells. Expected directions, top against bottom third (as HC
against LC): higher surround/center ratio, narrower temporal width, lower gain, higher offset.
Temporal width is from the frame-rate RFs here.

In [ ]:
offy = inc[(inc.onoff == "OFF") & (inc.xy == "Y")]
jobs5 = {}
for k, r in offy.iterrows():
    d = units[(r.subject, r.session, r.unit_name)]
    path = f"sub-{r.subject}/sub-{r.subject}_ses-{r.session}_ecephys.nwb"
    jobs5.setdefault(path, []).append((int(r.unit), d["window"], 5400 if r.subject == "Expt2" else 0, d["ns_H"]["rf"], d["ns_H"]["err"],
                                       [("a", "center"), ("b", "weighted")]))
t0 = time.time()
res5 = L.run_thirds(T, [(assets[p]["asset_id"], j, L.seed("fig5-within-trial-contrast")) for p, j in jobs5.items()])
print(f"{len(offy)} OFF-center Y cells, {len(jobs5)} penetrations, {time.time() - t0:.0f} s")
rows5 = []
for (p, js), out in zip(jobs5.items(), res5):
    for res in out:
        for key in ("a_center", "b_weighted"):
            d = res[key]
            rg = d["ranges"]
            inr = (np.abs(rg[:, 0] - 0.27) <= 0.027) & (np.abs(rg[:, 1] - 0.43) <= 0.043)
            top, bot = d["top"], d["bottom"]
            dirs = {"surround up": top["surround_ratio_a"] > bot["surround_ratio_a"], "temporal narrower": top["temporal_width"] < bot["temporal_width"],
                    "gain lower": top["gain"] < bot["gain"], "offset higher": top["offset"] > bot["offset"]}
            rows5.append({"file": p.split("/")[1].replace("_ecephys.nwb", ""), "unit": res["unit"], "reading": key, "mean contrast": d["mean"],
                          "thirds cuts": np.round(d["cuts"], 3).tolist(), "150 s min (lowest, median)": (round(rg[:, 0].min(), 3), round(float(np.median(rg[:, 0])), 3)),
                          "150 s max (median, highest)": (round(float(np.median(rg[:, 1])), 3), round(rg[:, 1].max(), 3)), "segments in tolerance": int(inr.sum()),
                          "surround top/bottom": (round(float(top["surround_ratio_a"]), 3), round(float(bot["surround_ratio_a"]), 3)),
                          "tw ms top/bottom": (round(float(top["temporal_width"]) * 1000, 1), round(float(bot["temporal_width"]) * 1000, 1)),
                          "gain top/bottom": (round(top["gain"], 5), round(bot["gain"], 5)), "offset top/bottom": (round(top["offset"]), round(bot["offset"])),
                          "spikes top/bottom": (int(top["n_spikes"]), int(bot["n_spikes"])), **dirs, "all four": all(dirs.values())})
f5 = pd.DataFrame(rows5)
print(f5.to_string(index=False))
for key, t in f5.groupby("reading"):
    print(f"reading {key}: cells with a 150 s segment in tolerance {int((t['segments in tolerance'] > 0).sum())} of {len(t)}; all four directions {int(t['all four'].sum())} of {len(t)}; "
          f"both {int(((t['segments in tolerance'] > 0) & t['all four']).sum())}; each direction: {t[['surround up', 'temporal narrower', 'gain lower', 'offset higher']].sum().to_dict()}")

**Result: not reproduced.** Among the 24 included OFF-center Y cells, none has a 150 s segment whose
contrast range is within tolerance of 0.27 to 0.43 under reading (a), where the per-frame contrast
reaches 0 in every cell and the median per-segment maximum is 1.24 to 2.46; under reading (b) 5
cells have such a segment. No cell shows all four directions under either reading. The top third
never has the lower gain (0 of 24 under both readings); its offset is higher for 15 (a) and 22 (b)
cells, its surround stronger for 0 (a) and 8 (b), and its temporal width narrower for 10 (a) and 4
(b).

The outcome rests on assumptions. Under contrast reading (a) the per-frame contrast runs far above
the paper's range, so its thirds are sorted largely by luminance; under reading (b) the thirds are
cut at about 0.35 and 0.38 (the printed cuts). Each third's RF and NL use the penalty of the full
data on one third of the frames. A variant with the penalty scaled to the third's frames was not run
(see the summary).

## Reliability and sparseness (Figure 6)

> fig6a-reliability-contrast: A decrease in contrast decreases response reliability by 54% for
> natural movies and 74% for white noise (31 cells). fig6a-reliability-correlations: a decrease in
> correlations decreases it by 68%.

> fig6b-sparseness-contrast: A decrease in contrast causes only a small decrease in sparseness for
> natural movies (4%) and a larger one for white noise (16%). fig6b-sparseness-correlations: a
> decrease in correlations decreases sparseness by 14%.

> disc-lcns-hcwn-reliability-sparseness: Low-contrast natural movies and high-contrast white noise
> evoke responses of similar reliability and sparseness, those to white noise slightly higher.

> disc-lcns-hcwn-rf-nl-differences: The differences in spatial width, surround strength, gain and
> normalized offset between LC NS and LC WN remain significant when LC NS is compared with HC WN.

Reliability readings P8: (a) the signal power includes the mean, (b) it does not. `decrease_a_%`
and `decrease_b_%` are P1 (a) and (b).

In [ ]:
rs = []
for rd, pre in [("a", "rel_a_"), ("b", "rel_b_")]:
    for a, b in [("ns_H", "ns_L"), ("wn_H", "wn_L"), ("ns_L", "wn_L"), ("ns_L", "wn_H")]:
        x, y = wn[pre + a], wn[pre + b]
        ok = np.isfinite(x) & np.isfinite(y)
        x, y = x[ok], y[ok]
        t = stats.ttest_rel(x, y)
        pc = L.pct_change(x, y)
        rs.append({"reading": rd, "comparison": f"{a}->{b}", "n": int(ok.sum()), "mean_a": x.mean(), "mean_b": y.mean(), "sd_a": x.std(), "sd_b": y.std(),
                   "decrease_a_%": -pc["a"], "decrease_b_%": -pc["b"], "t": t.statistic, "p": t.pvalue})
rel = pd.DataFrame(rs)
print("cells with a non-finite reliability (no spikes in a block):", [f"{r.session} {r.unit_name}" for r in wn.itertuples()
      if not all(np.isfinite([getattr(r, "rel_a_" + c) for c in L.CONDITIONS]))])
print(f"{len(wn)} cells; non-finite reliabilities:", {c: int((~np.isfinite(wn["rel_a_" + c].astype(float))).sum()) for c in L.CONDITIONS})
print(rel.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\nby penetration, reading a:")
print(wn.groupby("session")[["rel_a_ns_H", "rel_a_ns_L", "rel_a_wn_H", "rel_a_wn_L"]].mean().round(3).to_string())

**Result: partially reproduced.** Over 30 cells (one has a block without spikes), reliability falls
at low contrast for the movie (p = 3.1e-05) and the noise (p = 1.9e-09). The noise decrease, 70.2%
(P8 a) or 74.9% (P8 b) under P1 a, is within 66.6 to 81.4%. The movie decrease is 47.2% (P8 a) or
48.5% (P8 b), just below 48.6 to 59.4%.

In [ ]:
print(rel[rel.comparison == "ns_L->wn_L"].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

**Result: reproduced.** From LC NS to LC WN reliability falls by 62.7% (P8 a) or 67.7% (P8 b) under
P1 a, within 61.2 to 74.8%, paired t p = 1.7e-07 over 30 cells.

In [ ]:
sp = []
for a, b in [("ns_H", "ns_L"), ("wn_H", "wn_L"), ("ns_L", "wn_L"), ("ns_L", "wn_H")]:
    x, y = wn["sparse_" + a], wn["sparse_" + b]
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    t = stats.ttest_rel(x, y)
    pc = L.pct_change(x, y)
    sp.append({"comparison": f"{a}->{b}", "n": int(ok.sum()), "mean_a": x.mean(), "mean_b": y.mean(), "sd_a": x.std(), "sd_b": y.std(),
               "decrease_a_%": -pc["a"], "decrease_b_%": -pc["b"], "t": t.statistic, "p": t.pvalue})
spt = pd.DataFrame(sp)
print(spt.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\nby penetration:")
print(wn.groupby("session")[["sparse_ns_H", "sparse_ns_L", "sparse_wn_H", "sparse_wn_L"]].mean().round(3).to_string())

**Result: partially reproduced.** Sparseness falls slightly for the movie (2.2%, p = 0.023) and more
for the noise (15.7%, p = 7.8e-12), so the noise decrease exceeds the movie decrease. The noise
decrease is within 14.4 to 17.6%; the movie decrease is below the paper's 4% (3.6 to 4.4%).

In [ ]:
print(spt[spt.comparison == "ns_L->wn_L"].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

**Result: reproduced.** From LC NS to LC WN sparseness falls by 13.1% (P1 a and b), within 12.6 to
15.4%; paired t p = 1.3e-08.

In [ ]:
print(rel[rel.comparison == "ns_L->wn_H"].to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print(spt[spt.comparison == "ns_L->wn_H"].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

**Result: reproduced.** HC WN responses are more reliable (0.378 against 0.302, P8 a) and sparser
(0.856 against 0.831) than LC NS responses; neither is lower. Both differences are significant (p =
0.0058 and 0.00037), larger than the paper's "slightly higher".

In [ ]:
want = {"spatial_width": -1, "surround_ratio_a": -1, "gain": 1, "norm_offset_adj": 1}
t = cmp[(cmp.comparison == "ns_L->wn_H") & cmp.prop.isin(list(want) + ["surround_ratio_b"])].copy()
t["direction_ok"] = [np.sign(r.mean_b - r.mean_a) == want.get(r.prop, -1) for r in t.itertuples()]
print(t[["prop", "n", "mean_a", "mean_b", "pct_a", "t", "p", "direction_ok"]].round(4).to_string(index=False))
for lvl in (0.05, 0.001):
    m = t.prop.isin(want)
    print(f"level {lvl}: {int(((t.p < lvl) & t.direction_ok & m).sum())} of 4 significant in the direction of the LC NS -> LC WN change")
print("\nvariant, the first round's uneven offset adjustment, normalized offset LC NS -> HC WN:")
print(uneven[(uneven.prop == "norm_offset_adj") & (uneven.comparison == "ns_L->wn_H")][["prop", "n", "mean_a", "mean_b", "pct_a", "t", "p"]].round(4).to_string(index=False))

**Result: not reproduced.** Comparing LC NS with HC WN over 31 cells, the spatial width (-21.1%,
p = 1.2e-15) and the surround/center ratio (-38.3%, p = 6.3e-09) differ in the direction of the LC
NS to LC WN change. The normalized offset is higher for HC WN (+27.6%) with p = 0.056, and the gain
differs in the opposite direction (-29.4%, p = 0.00024): 2 of 4 at p < 0.05 and at p < 0.001. With
the first round's uneven offset adjustment the normalized offset gives p = 0.0048, which would make
3 of 4 at p < 0.05.

## Generic models (Figures 7 and 8)

> fig7c-effective-contrast: Filtering by the RF raises the effective contrast of the natural movies
> relative to white noise, so that low-contrast natural movies and high-contrast white noise have
> similar effective contrasts despite different RMS contrasts.

> fig7d: Normalized to high-contrast natural movies, the effective contrasts of the four stimuli
> correspond closely to the observed reliability of LGN responses.

> fig8a: The natural movies filtered by the generic RFs have heavy-tailed distributions (kurtosis
> 4.06) while filtered white noise is Gaussian (kurtosis 3.01), and shuffling the movie frames
> before filtering removes the excess kurtosis.

> fig8-model-sparseness-shuffle: The generic LN model with the high-contrast natural movie RF and NL
> gives a response sparseness of 0.86 to the movie and 0.83 when the movie frames are shuffled.

> fig8b: The sparseness of the noiseless generic LN model increases as contrast and correlations
> decrease, the opposite of the experimental responses, matching the experiment only for
> high-contrast natural movies.

> fig8de: Adding Gaussian white noise to the filtered stimulus before the nonlinearity lowers model
> sparseness most for low-contrast white noise, and at one noise level the model and experimental
> sparseness agree closely for all four stimuli.

Generic RF: the mean of the cells' RFs (OFF sign-reversed), normalized to unit variance. Cells:
(a) the 31 Expt1 cells for all four stimuli, (b) all included cells with each stimulus. Stimulus
filtered: (a) the long segments, (b) the 12 s repeat clips, at 16 window positions on a 9-pixel
grid of the 48 x 48 frame (a standard choice). Effective contrast is the SD of the filtered
stimulus; RMS contrast the SD of the stimulus, both in gray levels. The generic NL uses the mean
gain and the offset under readings (a) mean normalized offset x SD of the filtered stimulus, (b)
mean adjusted offset.

In [ ]:
def generic_rf(data, cond):
    rfs = [units[(r.subject, r.session, r.unit_name)][cond]["rf"] * (1 if r.onoff == "ON" else -1) for r in data.itertuples()]
    g = np.mean(rfs, 0)
    return g / g.std()
positions = [(r0, c0) for r0 in (0, 9, 18, 27) for c0 in (0, 9, 18, 27)]
def stim_long(cond):
    tm = "NAT" if cond.startswith("ns") else "WN"
    n = 54000 if tm == "NAT" else int(round(L.WN_SECONDS * L.FRAME_RATE))
    return T[tm][:n], {"ns_H": 0.40, "ns_L": 0.15, "wn_H": 0.55, "wn_L": 0.20}[cond]
def stim_clips(cond):
    k = "clip_nr_b" if cond.startswith("ns") else "clip_wr_b"
    return [clips[k + str(i)] for i in (1, 2, 3)], {"ns_H": 0.40, "ns_L": 0.15, "wn_H": 0.55, "wn_L": 0.20}[cond]
def filt(F, rms, g):
    return np.concatenate([L.filtered(np.ascontiguousarray(F[:, r0:r0 + L.WIN, c0:c0 + L.WIN]).reshape(len(F), -1) * np.float32(125 * rms), g)[L.NLAGS - 1:]
                           for r0, c0 in positions])
def filtered_stim(cond, g, which):
    if which == "a":
        F, rms = stim_long(cond)
        return filt(F, rms, g), 125 * rms * float(F.std())
    Fs, rms = stim_clips(cond)
    return np.concatenate([filt(F, rms, g) for F in Fs]), 125 * rms * float(np.concatenate(Fs).std())
groups = {"a": {c: wn for c in L.CONDITIONS}, "b": {"ns_H": inc, "ns_L": inc, "wn_H": wn, "wn_L": wn}}
G, Y, eff = {}, {}, []
for gr in ("a", "b"):
    for c in L.CONDITIONS:
        G[gr, c] = generic_rf(groups[gr][c], c)
        for st in ("a", "b"):
            y, rmsg = filtered_stim(c, G[gr, c], st)
            Y[gr, st, c] = y
            eff.append({"cells": gr, "stimulus": st, "cond": c, "n_cells": len(groups[gr][c]), "rms_contrast_gray": rmsg,
                        "effective": float(y.std()), "ratio": float(y.std()) / rmsg})
eff = pd.DataFrame(eff)
print(eff.round(3).to_string(index=False))
for (gr, st), t in eff.groupby(["cells", "stimulus"]):
    t = t.set_index("cond")
    ratio_ok = min(t.ratio["ns_H"], t.ratio["ns_L"]) > max(t.ratio["wn_H"], t.ratio["wn_L"])
    d = {c: abs(t.effective["ns_L"] - t.effective[c]) for c in ["ns_H", "wn_H", "wn_L"]}
    print(f"cells {gr}, stimulus {st}: NS ratios above WN ratios: {ratio_ok}; |LC NS - x| effective: "
          f"{ {k: round(v, 1) for k, v in d.items()} }; closest to HC WN: {min(d, key=d.get) == 'wn_H'}")
ok_cells = wn[wn.sign_ok]
ev = {c: float(filtered_stim(c, generic_rf(ok_cells, c), "a")[0].std()) for c in L.CONDITIONS}
rv = {c: ev[c] / filtered_stim(c, G["a", c], "a")[1] for c in L.CONDITIONS}
print(f"\nvariant, generic RFs from the {len(ok_cells)} of the 31 cells whose RFs keep their sign in every condition, long segments: "
      f"effective {dict((k, round(v)) for k, v in ev.items())}, ratio {dict((k, round(v, 1)) for k, v in rv.items())}")

**Result: reproduced.** Under every reading, the ratio of effective to RMS contrast is 243 to 277 for
the movies and 72 to 73 for the noise, and the LC NS effective contrast (4763 to 5185) is closest to
that of HC WN (4997 to 5010). Generic RFs from the 24 of the 31 cells whose RFs keep their sign give
the same picture (ratios 259 and 272 against 73).

In [ ]:
fd = []
for gr in ("a", "b"):
    for st in ("a", "b"):
        e = eff[(eff.cells == gr) & (eff.stimulus == st)].set_index("cond").effective
        for rd in ("a", "b"):
            m = wn[[f"rel_{rd}_{c}" for c in L.CONDITIONS]].mean().values
            s = wn[[f"rel_{rd}_{c}" for c in L.CONDITIONS]].std().values
            en = e[L.CONDITIONS].values / e["ns_H"]
            mn, sn = m / m[0], s / m[0]
            same_rank = list(np.argsort(en)) == list(np.argsort(mn))
            within = np.abs(en - mn) <= sn
            fd.append({"cells": gr, "stimulus": st, "reliability": rd, "eff_norm": np.round(en, 3).tolist(), "rel_norm": np.round(mn, 3).tolist(),
                       "rel_sd_norm": np.round(sn, 3).tolist(), "same_rank": same_rank, "n_within_sd": int(within.sum())})
print(pd.DataFrame(fd).to_string(index=False))

**Result: reproduced, under the 12 s clip reading of the stimulus.** Normalized to HC NS, the
effective contrasts are 1, 0.384, 0.404 and 0.146 (HC NS, LC NS, HC WN, LC WN) and the reliabilities
1, 0.528, 0.662 and 0.197: same ranks, each within one normalized SD. With the long segments the
effective contrasts of LC NS and HC WN swap order (0.390 and 0.382), so the ranks differ.

fig8a, fig8b and fig8de pass only with the long segments, the opposite stimulus reading, so no
single reading passes all four claims; this outcome rests on the 12 s clip reading.

In [ ]:
rng = np.random.default_rng(L.seed("fig8a-filtered-kurtosis"))
kt = []
for gr in ("a", "b"):
    for st in ("a", "b"):
        row = {"cells": gr, "stimulus": st}
        for c in L.CONDITIONS:
            row[c] = stats.kurtosis(Y[gr, st, c], fisher=False)
        kt.append(row)
kt = pd.DataFrame(kt)
print("kurtosis (Gaussian = 3) of the filtered stimuli:")
print(kt.round(3).to_string(index=False))
F, rms = stim_long("ns_H")
sh = []
for k in range(5):
    rng_k = np.random.default_rng(L.seed("fig8a-filtered-kurtosis") + 1000 * k)
    perm = rng_k.permutation(len(F))
    sh.append([stats.kurtosis(filt(F[perm], rms, G[gr, "ns_H"]), fisher=False) for gr in ("a", "b")])
sh = np.array(sh)
print("frame-shuffled HC NS filtered by the HC NS generic RF, kurtosis (cells a, b), claim seed:", np.round(sh[0], 3),
      "; over the five seeds:", np.round(sh.min(0), 3), "to", np.round(sh.max(0), 3))

**Result: reproduced.** Filtered with the generic RFs over the long segments, the movies have
kurtosis 4.01 (HC) and 3.79 (LC), within 3.65 to 4.47, and the noise 2.96 and 2.98, within 2.71 to
3.31. Frame-shuffled HC NS has kurtosis 3.18 (3.16 to 3.19 over five seeds), within 0.30 of 3. Over
the 12 s clips the movie values are lower (3.44 to 3.55).

In [ ]:
def model_sparse(y, gain, q):
    return L.sparseness(gain * np.maximum(y - q, 0))
gm = []
for gr in ("a", "b"):
    for st in ("a", "b"):
        for rd in ("a", "b"):
            row = {"cells": gr, "stimulus": st, "offset": rd}
            for c in L.CONDITIONS:
                data = groups[gr][c]
                y = Y[gr, st, c]
                q = data[c + "_norm_offset_adj"].mean() * y.std() if rd == "a" else data[c + "_offset_adj"].mean()
                row[c] = model_sparse(y, data[c + "_gain"].mean(), q)
            gm.append(row)
gm = pd.DataFrame(gm)
print("model sparseness:")
print(gm.round(3).to_string(index=False))
F, rms = stim_long("ns_H")
for gr in ("a", "b"):
    data = groups[gr]["ns_H"]
    vals = []
    for k in range(5):
        perm = np.random.default_rng(L.seed("fig8-model-sparseness-shuffle") + 1000 * k).permutation(len(F))
        y = filt(F[perm], rms, G[gr, "ns_H"])
        vals.append([model_sparse(y, data.ns_H_gain.mean(), data.ns_H_norm_offset_adj.mean() * y.std()),
                     model_sparse(y, data.ns_H_gain.mean(), data.ns_H_offset_adj.mean())])
    vals = np.array(vals)
    print(f"cells {gr}: frame-shuffled HC NS model sparseness, offset a {vals[0, 0]:.3f}, offset b {vals[0, 1]:.3f} (claim seed); "
          f"over five seeds {np.round(vals.min(0), 3)} to {np.round(vals.max(0), 3)}")

**Result: reproduced.** The generic model with the HC NS RF and NL gives sparseness 0.805 for the
movie and 0.773 for the frame-shuffled movie (offset reading a, 31 cells, long segments), within the
tolerances of 0.86 and 0.83, and the shuffled value is lower. The shuffled value is 0.773 under all
five seeds.

In [ ]:
exp_m = wn[[f"sparse_{c}" for c in L.CONDITIONS]].mean().values
exp_s = wn[[f"sparse_{c}" for c in L.CONDITIONS]].std().values
print("experimental sparseness (12 s repeats):", dict(zip(L.CONDITIONS, np.round(exp_m, 3))), "SD", dict(zip(L.CONDITIONS, np.round(exp_s, 3))))
for r in gm.itertuples():
    mv = np.array([getattr(r, c) for c in L.CONDITIONS])
    print(f"cells {r.cells} stim {r.stimulus} offset {r.offset}: lowest {L.CONDITIONS[int(np.argmin(mv))]}, highest {L.CONDITIONS[int(np.argmax(mv))]}, "
          f"HC NS model {mv[0]:.3f} within 1 SD of experiment: {abs(mv[0] - exp_m[0]) <= exp_s[0]}")
print("experiment HC NS above LC WN:", exp_m[0] > exp_m[3])

**Result: reproduced.** With the 31 cells, the long segments and offset reading (a), the noiseless
model is least sparse for HC NS (0.805) and most for LC WN (0.851); the experiment is sparser for HC
NS (0.850) than LC WN (0.722); and the model HC NS value is within one SD (0.049) of the experiment.
Under the 12 s clips the model HC NS value (0.791) is outside one SD.

In [ ]:
# Noise added to the filtered stimulus before the NL (Figure 8C), cells reading a, offset reading a,
# under both stimulus readings: (a) the long segments, (b) the 12 s repeat clips. Noise SD in
# filtered-stimulus units. For the long segments, reading (b) (120 noisy trials) uses every k-th
# filtered value, at most 60000, to bound the run time (a standard choice: sparseness is a
# distribution statistic).
levels = np.linspace(0, 6000, 41)
def noisy_sparse(c, st, sd_noise, rng, trials):
    data = groups["a"][c]
    y = Y["a", st, c]
    q = data[c + "_norm_offset_adj"].mean() * y.std()
    if trials == 1:
        return model_sparse(y + rng.standard_normal(len(y)) * sd_noise, data[c + "_gain"].mean(), q)
    y = y[::int(np.ceil(len(y) / 60000))]
    acc = np.zeros(len(y))
    for _ in range(trials):
        acc += data[c + "_gain"].mean() * np.maximum(y + rng.standard_normal(len(y)) * sd_noise - q, 0)
    return L.sparseness(acc / trials)
def scan(st, rd, k):
    trials = 1 if rd == "a" else 120
    rng = np.random.default_rng(L.seed("fig8de-noise-reconciles") + {"aa": 0, "ab": 1, "ba": 2, "bb": 3}[st + rd] + k)
    return np.array([[noisy_sparse(c, st, s, rng, trials) for c in L.CONDITIONS] for s in levels])
out8 = {}
for st in ("a", "b"):
    for rd in ("a", "b"):
        M = scan(st, rd, 0)
        out8[st, rd] = M
        within = (np.abs(M - exp_m) <= exp_s).all(1)
        print(f"stimulus {st}, reading {rd}: decrease from no noise to the largest level: {dict(zip(L.CONDITIONS, np.round(M[0] - M[-1], 3)))}; "
              f"largest for {L.CONDITIONS[int(np.argmax(M[0] - M[-1]))]}; levels with all four within 1 SD: {levels[within].tolist()}")
        print("   per stimulus, levels within 1 SD:", {c: levels[np.abs(M[:, j] - exp_m[j]) <= exp_s[j]].tolist()[:3] + (["..."] if (np.abs(M[:, j] - exp_m[j]) <= exp_s[j]).sum() > 3 else []) for j, c in enumerate(L.CONDITIONS)})
        print(pd.DataFrame(M[::5], index=levels[::5], columns=L.CONDITIONS).round(3).to_string())
for k in (1000, 2000, 3000, 4000):
    for st in ("a", "b"):
        for rd in ("a", "b"):
            M = scan(st, rd, k)
            print(f"seed + {k}, stimulus {st}, reading {rd}: largest decrease for {L.CONDITIONS[int(np.argmax(M[0] - M[-1]))]}; "
                  f"levels with all four within 1 SD: {int((np.abs(M - exp_m) <= exp_s).all(1).sum())}; HC NS model max {M[:, 0].max():.3f}")

**Result: reproduced, under stimulus reading (a).** With the long segments, added noise lowers model
sparseness most for LC WN under both readings (decrease 0.115 against 0.020 to 0.066 with one
realization; 0.693 against 0.064 to 0.306 with 120 trials), and at noise SDs of 1800 to 2100 (one
realization) or 750 to 1350 (120 trials) all four model values lie within one SD of the experiment;
the other four seeds give 3 and 5 such levels. With the 12 s clips (stimulus reading b) no level
puts all four within one SD, as the HC NS model value is at most 0.791, below 0.850 - 0.049.

fig7d passes only with the 12 s clips, the opposite stimulus reading; this outcome rests on the
long-segment reading.

## Claims from the Experimental Procedures

> methods-latency-biphasic: Changes in contrast and correlations significantly change the latency
> (time to peak) of the RF center's temporal profile but not its biphasic ratio.

> methods-offset-varies-across-animals: Of the RF and NL parameters, only the offset q differs
> significantly across animals within a stimulus condition.

Latency and biphasic ratio from the 128 Hz RFs (8 ms lags); the frame-rate values (16.7 ms lags,
latency in steps of one frame) are a variant.

In [ ]:
for props in (["latency_8", "biphasic_8"], ["latency", "biphasic"]):
    lt = cmp[cmp.prop.isin(props) & cmp.comparison.isin(["ns_H->ns_L", "wn_H->wn_L", "ns_L->wn_L"])]
    lt = lt[~((lt.comparison == "ns_H->ns_L") & (lt.group != f"all {len(inc)}"))]
    print("main (128 Hz RFs):" if props[0].endswith("_8") else "\nvariant (frame-rate RFs):")
    print(lt[["prop", "comparison", "n", "mean_a", "mean_b", "t", "p"]].round(4).to_string(index=False))
    for lvl in (0.05, 0.001):
        print(f"latency significant at {lvl}: {int((lt[lt.prop == props[0]].p < lvl).sum())} of 3; biphasic significant at 0.05: {int((lt[lt.prop == props[1]].p < 0.05).sum())} of 3")

**Result: reproduced.** With the 128 Hz RFs, latency changes in all three comparisons at p < 0.05:
HC to LC movie 36.5 to 41.3 ms (111 cells, t = -5.67, printed p 0.0000), HC to LC noise 41.7 to
53.8 ms (p = 0.0061), LC movie to LC noise 42.5 to 53.8 ms (p = 0.035); at p < 0.001 only the first.
The biphasic ratio changes in none (p = 0.13 to 0.92). With the frame-rate RFs (variant) latency
changes in 2 of 3 (movie p = 0.14).

In [ ]:
an = []
for c in ["ns_H", "ns_L"]:
    for prop in ["spatial_width", "surround_ratio_a", "temporal_width_8", "gain", "offset", "norm_offset"]:
        g1, g2 = [inc[inc.subject == s][c + "_" + prop].dropna() for s in ("Expt1", "Expt2")]
        an.append({"cond": c, "prop": prop, "mean_Expt1": g1.mean(), "mean_Expt2": g2.mean(), "anova_p": stats.f_oneway(g1, g2).pvalue,
                   "kruskal_p": stats.kruskal(g1, g2).pvalue})
an = pd.DataFrame(an)
print(an.round(4).to_string(index=False))
for lvl in (0.05, 0.001):
    for t in ("anova_p", "kruskal_p"):
        sig = an[an[t] < lvl]
        print(f"{t} < {lvl}: offset in {int((sig.prop == 'offset').sum())} conditions; other parameters significant: {sorted(set(sig.prop) - {'offset'})}")
print("White noise was recorded in one animal (Expt1) only, so no test across animals is possible for it.")

**Result: not reproduced.** Between Expt1 and Expt2 the offset differs at low contrast (ANOVA
p = 0.021) but not at high contrast (p = 0.46), and spatial width, surround/center ratio, temporal
width and gain also differ at p < 0.05 (ANOVA p from below 0.0001 to 0.047). At p < 0.001 the offset
differs in no condition, while surround/center ratio and temporal width do. Only two animals can be
compared, and only for the movie.

## Cell types

> celltype-y-wider-than-x: The spatial RF center is wider for Y than X cells during high-contrast
> (0.75 against 0.63 deg) and low-contrast (0.73 against 0.63 deg) natural movies.

> celltype-y-stronger-surround: The relative surround strength is larger for Y than X cells during
> high-contrast natural movies (0.16 against 0.13).

> celltype-off-wider-temporal: During low-contrast white noise the temporal profile of the RF
> center is wider for OFF than ON cells (29 against 25 ms).

> celltype-prediction-x-y: LN model predictions are slightly more accurate for X than Y cells at
> high (0.73 against 0.69) and low (0.78 against 0.75) contrast, but the differences are not
> significant.

In [ ]:
xy = inc[inc.xy.isin(["X", "Y"])]
print("X and Y cells:", xy.xy.value_counts().to_dict(), "; by subject:", xy.groupby(["subject", "xy"]).size().to_dict())
for c in ["ns_H", "ns_L"]:
    a, b = xy[xy.xy == "X"][c + "_spatial_width"].dropna(), xy[xy.xy == "Y"][c + "_spatial_width"].dropna()
    t = stats.ttest_ind(a, b)
    print(f"{c}: X {a.mean():.3f} ({a.std():.3f}, n={len(a)}), Y {b.mean():.3f} ({b.std():.3f}, n={len(b)}) deg; t = {t.statistic:.2f}, p = {t.pvalue:.3g}")

**Result: reproduced.** The spatial width is larger for Y than X cells at high contrast (0.743
against 0.646 deg, p = 0.00086) and low contrast (0.778 against 0.642 deg, p = 0.00013), all four
means within the paper's SDs.

In [ ]:
for rd in ("a", "b"):
    a, b = xy[xy.xy == "X"][f"ns_H_surround_ratio_{rd}"].dropna(), xy[xy.xy == "Y"][f"ns_H_surround_ratio_{rd}"].dropna()
    t = stats.ttest_ind(a, b)
    print(f"reading {rd}: X {a.mean():.3f} ({a.std():.3f}, n={len(a)}), Y {b.mean():.3f} ({b.std():.3f}, n={len(b)}); t = {t.statistic:.2f}, p = {t.pvalue:.3g}")

**Result: not reproduced.** At high contrast the surround/center ratio is 0.105 for X and 0.113 for Y
cells, both within the paper's SDs, but the difference is not significant (p = 0.27).

In [ ]:
for name, data in [("all 31 cells", wn), ("variant: cells whose RFs keep their sign in every condition", wn[wn.sign_ok])]:
    for tw in ("wn_L_temporal_width_8", "wn_L_temporal_width"):
        a, b = data[data.onoff == "ON"][tw].dropna() * 1000, data[data.onoff == "OFF"][tw].dropna() * 1000
        t = stats.ttest_ind(a, b)
        print(f"{name}, {'128 Hz' if tw.endswith('_8') else 'frame-rate (variant)'}: LC WN temporal width ON {a.mean():.1f} (SD {a.std():.1f}, SE {a.sem():.1f}, n={len(a)}), "
              f"OFF {b.mean():.1f} (SD {b.std():.1f}, SE {b.sem():.1f}, n={len(b)}) ms; t = {t.statistic:.2f}, p = {t.pvalue:.3g}")

**Result: not reproduced.** With the 128 Hz RFs the LC WN temporal width is 33.5 ms for ON (n = 17)
and 31.5 ms for OFF cells (n = 14), p = 0.71, against 25 and 29 ms; with the frame-rate RFs 37.0 and
35.9 ms, p = 0.84. Leaving out the 7 cells whose RFs change sign between conditions gives 30.0 and
28.7 ms (p = 0.75) with the 128 Hz RFs; with the frame-rate RFs OFF is then wider, 33.2 against
27.7 ms (p = 0.0074), but 33.2 is outside 29 +- 2.9.

In [ ]:
pxy = pr[pr.xy.isin(["X", "Y"])]
print("classified cells of the prediction set:", pxy.xy.value_counts().to_dict())
for rd in ("a", "b"):
    for c in ("H", "L"):
        a, b = pxy[pxy.xy == "X"][f"{rd}_{c}"], pxy[pxy.xy == "Y"][f"{rd}_{c}"]
        t = stats.ttest_ind(a, b)
        print(f"reading {rd}, {c}C: X {a.mean():.3f} (n={len(a)}), Y {b.mean():.3f} (n={len(b)}); t = {t.statistic:.2f}, p = {t.pvalue:.3g}")

**Result: not reproduced.** The X and Y predictions do not differ (p = 0.79 and 0.51, reading a), as
the paper says, but the means (0.471 and 0.482 at HC, 0.516 and 0.548 at LC) are far below 0.73, 0.69,
0.78 and 0.75, and Y is not below X.

## methods-spherical-asymmetry

> RFs estimated by the paper's least-squares method from simulated LN responses to the movies do
> not differ significantly from those estimated from simulated responses to Gaussian white noise,
> so the movies' spherical asymmetry does not bias the RF estimates much.

The simulation the paper describes ("we simulated LGN responses using the LN model with a known RF
and NL", Estimation of Receptive Fields, paragraph 3) with the known RF under readings (a) the
generic HC NS RF (31 cells), (b) the frame-rate HC NS RF of the included cell with the largest
peak/noise among the 31. NL: half-wave rectifier with the 31 cells' mean HC NS gain and normalized
offset (offset reading a of Figure 8). Stimuli: the 15 min movie at HC (a central 21 x 21 window,
rows and columns 13 to 33) and Gaussian white noise of the same length and SD (50 gray levels).
Spike counts per frame are Poisson with the model rate. RFs are estimated as for the cells (nine
segments, the chosen penalty, frame rate), with the same number of spikes for movie and noise (the
movie response is cut to the noise's spike count, as the paper does for HC and LC). As a reference
for chance, a second movie-driven simulation (new Poisson draws, same frames, cut to the same
number of spikes) is compared with the first under the same rules. The criterion's "within the spread of the nine segment
estimates at every RF point" is read as (i) the two estimates' one-SD bands overlap
(|movie - noise| <= SD movie + SD noise) and, stricter, (ii) |movie - noise| <= the larger SD. The
known RF's correlation with each estimate is also given.

In [ ]:
def simulate(g, S, gain, qn, rng):
    y = L.filtered(S, g)
    rate = gain * np.maximum(y - qn * np.nanstd(y), 0)
    rate[~np.isfinite(rate)] = 0
    return rng.poisson(rate / L.FRAME_RATE).astype(float)
def agree(m, sm, w, sw):
    d = np.abs(m - w)
    return {"share (i) overlap": float((d <= sm + sw).mean()), "points outside (i)": int((d > sm + sw).sum()),
            "share (ii) larger SD": float((d <= np.maximum(sm, sw)).mean()), "points outside (ii)": int((d > np.maximum(sm, sw)).sum()),
            "r": np.corrcoef(m.ravel(), w.ravel())[0, 1]}
Fm = np.ascontiguousarray(T["NAT"][:, 13:13 + L.WIN, 13:13 + L.WIN]).reshape(len(T["NAT"]), -1) * np.float32(125 * 0.40)
gain_g, qn_g = wn.ns_H_gain.mean(), wn.ns_H_norm_offset_adj.mean()
best = wn.ns_H_snr.idxmax()
known = {"a": G["a", "ns_H"], "b": units[(wn.subject[best], wn.session[best], wn.unit_name[best])]["ns_H"]["rf"]}
known["b"] = known["b"] / known["b"].std()
print(f"reading (b) cell: {wn.session[best]} {wn.unit_name[best]}, peak/noise {wn.ns_H_snr[best]:.1f}; gain {gain_g:.4g}, normalized offset {qn_g:.3f}")
sph = []
for k in (0, 1000, 2000, 3000, 4000):
    rng = np.random.default_rng(L.seed("methods-spherical-asymmetry") + k)
    Fw = (rng.standard_normal(Fm.shape) * Fm.std()).astype(np.float32)
    for rd in ("a", "b"):
        g = known[rd]
        rw, rm, rm2 = simulate(g, Fw, gain_g, qn_g, rng), simulate(g, Fm, gain_g, qn_g, rng), simulate(g, Fm, gain_g, qn_g, rng)
        n = min(rw.sum(), rm.sum(), rm2.sum())
        ends = [int(np.searchsorted(np.cumsum(r), n)) + 1 for r in (rw, rm, rm2)]
        (w, sw), (m, sm), (m2, sm2) = [(e["rf"], e["sd"]) for e in
                                       (L.estimate_rf(S, r, 0, min(b, len(r)))[0] for S, r, b in ((Fw, rw, ends[0]), (Fm, rm, ends[1]), (Fm, rm2, ends[2])))]
        a1, a2 = agree(m, sm, w, sw), agree(m, sm, m2, sm2)
        sph.append({"seed": f"+{k}", "known RF": rd, "spikes": int(n), "frames movie": ends[1], "frames noise": ends[0],
                    "movie-noise outside (i)": a1["points outside (i)"], "movie-noise outside (ii)": a1["points outside (ii)"],
                    "movie-movie outside (i)": a2["points outside (i)"], "movie-movie outside (ii)": a2["points outside (ii)"],
                    "r(movie, noise)": a1["r"], "r(movie, movie)": a2["r"],
                    "r(known, movie)": np.corrcoef(g.ravel(), m.ravel())[0, 1], "r(known, noise)": np.corrcoef(g.ravel(), w.ravel())[0, 1]})
    del Fw
sph = pd.DataFrame(sph)
print(f"RF points: {m.size}")
print(sph.round(4).to_string(index=False))

**Result: not reproduced.** For both known RFs and all five seeds, the movie-driven and noise-driven
estimates (same number of spikes, 2134 to 2383) differ by more than the sum of their segment SDs at
32 to 66 of the 5292 RF points (reading i), and by more than the larger SD at 357 to 422 points
(reading ii), so they do not agree at every point. Two movie-driven simulations with new Poisson
draws differ at 1 to 3 points (i) and 68 to 113 points (ii), so the movie and noise estimates
differ more than chance alone gives. The movie estimates correlate more with the known RF (0.67 to
0.78) than the noise estimates (0.63 to 0.65).

The outcome rests on the penalty. It is fixed relative to the stimulus variance, and at equal spike
counts the movie estimates use 14203 to 19589 frames against 53935 to 54000 for the noise, so the
movie estimate is regularized about 2.8 to 3.8 times more heavily relative to its data. The
"beyond chance" comparison depends on this. A variant with the noise penalty scaled to its frame
count was not run (see the summary).

## Ridge penalty variants

The analysis above uses the penalty chosen by the paper's rule (held-out prediction of the movie).
Here the inclusion counts and the RF and NL comparisons are repeated with the two variant penalties
computed in the same pass: `rel1000`, 1000 x stimulus variance (below the mean eigenvalue of X'X,
so the movie estimate is not pulled towards the spike-triggered average), and `paper`, the paper's
literal penalty of 1000 in gray levels. The comparisons use the cells included under the main
penalty, with each variant's RFs and NLs, and the same offset adjustment.

In [ ]:
def variant_tables(w):
    d = dfv[w].copy()
    d.index = df.index
    ia = (d.ns_H_snr >= 10) & (d.ns_L_snr >= 10)
    ib = (d.ns_H_snr >= 10) | (d.ns_L_snr >= 10)
    iv = adjust(d[inc_a], ["ns_H", "ns_L"])
    wv = adjust(d[inc_a & (d.subject == "Expt1") & d.session.isin(["a11", "a13", "a14"])], L.CONDITIONS)
    rows = []
    for prop in ["spatial_width", "surround_ratio_a", "surround_ratio_b", "temporal_width", "gain", "offset_adj", "norm_offset_adj"]:
        rows.append(compare(prop, "ns_H", "ns_L", iv))
        for a, b in [("wn_H", "wn_L"), ("ns_L", "wn_L"), ("ns_L", "wn_H")]:
            rows.append(compare(prop, a, b, wv))
    return ia, ib, pd.DataFrame(rows), d
for w in dfv:
    ia, ib, t, d = variant_tables(w)
    print(f"=== penalty {w}: included at both contrasts {int(ia.sum())}, at either {int(ib.sum())}; with the long noise {int((ia & (d.subject == 'Expt1') & d.session.isin(['a11', 'a13', 'a14'])).sum())}; "
          f"Expt2 {int((ia & (d.subject == 'Expt2')).sum())}")
    print("peak/noise percentiles HC:", np.round(np.percentile(d.ns_H_snr, [5, 50, 95]), 1), "LC:", np.round(np.percentile(d.ns_L_snr, [5, 50, 95]), 1),
          "; median centre size (pixels) HC NS:", float(d.ns_H_n_center[inc_a].median()), "; median HC NS latency (ms):", round(float(d.ns_H_latency[inc_a].median()) * 1000, 1))
    print(f"comparisons over the cells included under the main penalty ({int(inc_a.sum())}; 31 with noise):")
    print(t[["prop", "comparison", "n", "mean_a", "mean_b", "pct_a", "pct_b", "ratio_c", "p"]].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

**Variants.** With 1000 x stimulus variance, 29 cells pass the inclusion rule at both contrasts (69
at either); with the paper's literal penalty none do (4 at either): its RFs are at the noise level,
with median peak/noise 4.1 (HC) and 3.9 (LC), a median centre of 1 pixel and a median HC latency of
158 ms. Over the cells included under the chosen penalty, 1000 x variance changes several results:
the width decrease from LC NS to LC WN is 5.1% (p = 0.11) in place of 18.9%; the surround decrease
with correlations is 2.7% (p = 0.67, reading a) in place of 38.7%; the movie surround decrease with
contrast is 17.5% (p = 4.8e-06), within the paper's 17%, while the noise surround still does not
change (p = 0.83); the offset does not differ between LC NS and LC WN (p = 0.30); and the normalized
offset changes in none of the 31-cell comparisons (p = 0.31 to 0.91). Gain still rises at low
contrast (149.8% and 126.2%, P1 a). The outcomes that rest on the chosen penalty are marked as
assumptions in the attempt record.

The 128 Hz RFs (temporal width, latency, biphasic ratio and their surround) were not re-estimated
under a variant penalty (see the summary).

## Summary

| Claim | Outcome |
|---|---|
| sample-composition | not reproduced |
| firing-rate-contrast | partially reproduced |
| firing-rate-lc-ns-wn-similar | reproduced (assumption) |
| ln-prediction-accuracy | not reproduced |
| fig3b-spatial-width-contrast | reproduced |
| fig3b-spatial-width-correlations | reproduced (assumption; not reproduced proposed) |
| fig3c-surround-contrast | not reproduced |
| fig3c-surround-correlations | reproduced (assumption; not reproduced proposed) |
| fig3d-temporal-width-contrast | partially reproduced (assumption; not reproduced proposed) |
| fig3d-temporal-width-correlations | not reproduced |
| surround-temporal-correlation | partially reproduced (assumption; reproduced proposed) |
| figs1-property-correlations | not reproduced |
| fig4b-gain-contrast | partially reproduced |
| fig4b-gain-correlations | partially reproduced (assumption) |
| fig4c-offset-contrast | reproduced |
| fig4c-offset-correlations-none | not reproduced (assumption) |
| fig4d-normalized-offset-contrast | not reproduced |
| fig4d-normalized-offset-correlations | not reproduced |
| fig5-within-trial-contrast | not reproduced (assumption) |
| fig6a-reliability-contrast | partially reproduced |
| fig6a-reliability-correlations | reproduced |
| fig6b-sparseness-contrast | partially reproduced |
| fig6b-sparseness-correlations | reproduced |
| fig7c-effective-contrast | reproduced |
| fig7d-reliability-tracks-effective-contrast | reproduced (assumption) |
| fig8a-filtered-kurtosis | reproduced |
| fig8-model-sparseness-shuffle | reproduced |
| fig8b-model-vs-experiment-sparseness | reproduced |
| fig8de-noise-reconciles | reproduced (assumption) |
| disc-lcns-hcwn-reliability-sparseness | reproduced |
| disc-lcns-hcwn-rf-nl-differences | not reproduced (assumption) |
| methods-latency-biphasic | reproduced (assumption) |
| methods-offset-varies-across-animals | not reproduced |
| celltype-y-wider-than-x | reproduced |
| celltype-y-stronger-surround | not reproduced |
| celltype-off-wider-temporal | not reproduced |
| celltype-prediction-x-y | not reproduced |
| methods-spherical-asymmetry | not reproduced (assumption) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The paper's own ridge penalty (D = 0.001, a penalty
of 1000 in gray levels) gives RFs at the noise level here and no cell passes the inclusion rule, so
the penalty is chosen by the rule the paper says it used (held-out prediction of the movie), 10000
times the stimulus variance; several RF results change under a lower penalty (section "Ridge
penalty variants"), and the outcomes that rest on it are marked as assumptions. Spatial width,
surround and the NLs come from RFs with 12 lags of one frame (16.7 ms); temporal width, latency and
biphasic ratio from RFs at 128 Hz with 24 lags of 8 ms, as in the paper, but in a 15 x 15 window in
place of 21 x 21. The paper's 68-cell sample cannot be identified: the inclusion rule passes 111
cells here, and every comparison uses those. The 90 s test segment of Expt2 is part of the 15 min
movie, so its frames were left out of the training data. The first 6 min of the longer noise
recordings are used. ON or OFF is taken from the sign of the RF. The offset is adjusted per animal
within each set of cells compared, so the 31 white-noise cells, all from Expt1, are not adjusted.
Figure 5 was tested on the OFF-center Y cells only, and the spherical-asymmetry simulation at the
frame rate.

**After the second verification.** Four changes of outcome are proposed and wait for a third
round: fig3b-spatial-width-correlations and fig3c-surround-correlations to not reproduced (under
1000 x variance neither effect holds, and the verifier's synthetic check gives both effects from
the estimator alone), fig3d-temporal-width-contrast to not reproduced (the noise test misses
p < 0.001, and the criterion's partial outcome requires the tests to hold), and
surround-temporal-correlation to reproduced under the stated reading that both measures come from
the 128 Hz RFs. fig7d and fig8de pass under opposite stimulus readings (12 s clips and long segments)
and are marked as assumptions, as are fig5 and methods-spherical-asymmetry. Three variants the
verifier asked for were not run: a second penalty for the 128 Hz RFs, a noise penalty scaled to its
frame count in the spherical-asymmetry simulation, and a penalty scaled to each third's frames in
Figure 5. With the 128 Hz variant the main pass ran past its 3600 s cell limit on this run's machine,
which is slower than the one of the verified run, and with the full notebook near the 60 min limit
there was no time left for the other two.

**What was found about the dataset.** The data the paper needs are there for two animals (Expt1,
Expt2), with the stimulus frames, and the clocks agree (the 90 s repeats correlate with the same
frames of the unique movie at zero shift). The 90 s cross-validation segment is the first 90 s of the
15 min movie, not a different segment as the paper says; the noise recordings last 10 and 15 min, not
6; the units carry no ON/OFF label; and nothing marks which units or penetrations the paper used.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")